# Silver — Cross-Table Validation

Read-only checks over published Silver tables, driven by `00_Silver_Registry`.

**Asserted (fails the run):** table exists; row preservation versus Bronze for row-preserving
tables; no NULL grain keys; unique grain key where profiling established uniqueness; every
registered DQ flag column exists, is boolean and has no NULLs.

In [0]:
%run ./00_Silver_Registry

In [0]:
import re
from pyspark.sql import functions as F

dbutils.widgets.text("source_schema", "cdr_mimiciv.bronze")
dbutils.widgets.text("target_schema", "cdr_mimiciv.silver")
dbutils.widgets.text("tables", "all")
dbutils.widgets.text("enforce_profiled_counts", "true")

BRONZE = dbutils.widgets.get("source_schema").strip()
SILVER = dbutils.widgets.get("target_schema").strip()
RAW_TABLES = dbutils.widgets.get("tables").strip()

for _schema in (BRONZE, SILVER):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*", _schema):
        raise ValueError(f"Schema must be catalog.schema: {_schema}")

tables = [t for t in REGISTRY if t not in EXCLUDED and t not in DEFERRED] if RAW_TABLES == "all" \
    else [t.strip() for t in RAW_TABLES.split(",") if t.strip()]
unknown = [t for t in tables if t not in REGISTRY]
if unknown:
    raise ValueError(f"Not in the Silver registry: {unknown}")

In [0]:
failures, report = [], []

for table in tables:
    spec = REGISTRY[table]
    target = f"{SILVER}.{table}"
    if not spark.catalog.tableExists(target):
        failures.append(f"{table}: Silver table missing")
        continue

    silver = spark.table(target)
    aggregates = [F.count(F.lit(1)).alias("rows")]
    for key in spec["keys"]:
        aggregates.append(F.sum(F.when(F.col(key).isNull(), 1).otherwise(0)).alias(f"null_{key}"))
    if spec["unique_key"] and len(spec["keys"]) == 1:
        aggregates.append(F.countDistinct(spec["keys"][0]).alias("distinct_key"))
    for flag_column in spec["flags"]:
        aggregates.append(F.sum(F.when(F.col(flag_column).isNull(), 1).otherwise(0)).alias(f"null_{flag_column}"))
    # one pass over the Silver table
    stats = silver.agg(*aggregates).first().asDict()

    if spec["row_preserving"]:
        bronze_rows = spark.table(f"{BRONZE}.{table}").count()   # Delta statistics
        if stats["rows"] != bronze_rows:
            failures.append(f"{table}: Silver rows {stats['rows']:,} != Bronze rows {bronze_rows:,}")
    for key in spec["keys"]:
        if stats[f"null_{key}"]:
            failures.append(f"{table}: {stats[f'null_{key}']:,} NULL {key}")
    if "distinct_key" in stats and stats["distinct_key"] != stats["rows"]:
        failures.append(f"{table}: {spec['keys'][0]} is not unique ({stats['distinct_key']:,} of {stats['rows']:,})")

    column_types = {f.name: f.dataType.simpleString() for f in silver.schema.fields}
    for flag_column in spec["flags"]:
        if column_types.get(flag_column) != "boolean":
            failures.append(f"{table}: {flag_column} is {column_types.get(flag_column)}, expected boolean")
        elif stats[f"null_{flag_column}"]:
            failures.append(f"{table}: {flag_column} contains NULLs")

    report.append((table, stats["rows"], {c: int(stats[f"null_{c}"]) for c in spec["keys"]}))

for table, rows, null_keys in report:
    print(f"{table:<18} rows={rows:>14,}  key NULLs={null_keys}")

In [0]:
if failures:
    print("\nCross-table validation FAILURES:")
    for failure in failures:
        print(" -", failure)
    raise AssertionError(f"{len(failures)} Silver cross-table validation failure(s)")

print("\nCross-table validation passed for:", tables)
dbutils.notebook.exit(f"cross_validation:OK:{len(tables)}")